**Table of contents**<a id='toc0_'></a>    
- [Photoswitching fingerprints: 1 fluorophore](#toc1_)    
  - [Data generation](#toc1_1_)    
  - [Reading and fitting the data](#toc1_2_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# <a id='toc1_'></a>[Photoswitching fingerprints: 1 fluorophore](#toc0_)

In [ ]:
import glob

import numpy as np
import pandas as pd

import fluopy.fitting as fit
import fluopy.fluorophores as fl
import fluopy.routines as rt
import fluopy.transitions as tr

%load_ext autoreload
%autoreload 2

saving_at = r"D:\python_output\Chapter_I\0_4_multi_f_PFA\1F"

## <a id='toc1_1_'></a>[Data generation](#toc0_)

In [ ]:
def prepare_transition_set():
    fluorophores = fl.construct_fluorophores(name="cy5_dna", count=1, shape="square")
    fluorophore_system = fl.FluorophoreSystem(fluorophores=fluorophores)
    transitions = fluorophore_system.load_transitions(
        bleaching=True,
        summarize=True,
        energy_transfer=True,
        **rt.PARAMS_DSTORM,
    )
    transition_set = tr.TransitionSet(transitions, fluorophore_system)
    transition_set.finalize()

    return transition_set

In [ ]:
rng = np.random.default_rng(1)
transition_set = prepare_transition_set()
_, _, _ = rt.fingerprint_analysis(
    transition_set,
    batch_size=40,
    batches=1,
    filepath=saving_at,
    filename="no_distance",
    seed=rng,
    use_memmap=r"C:\Users\vie43sq\Desktop\Simulations\memmaps\run_3",
)

## <a id='toc1_2_'></a>[Reading and fitting the data](#toc0_)

In [ ]:
x = np.linspace(0, 300, 300001)
rng = np.random.default_rng(1)
fingerprints = pd.Series(np.zeros(300001), np.round(x, decimals=12), dtype=np.int32)

for file in glob.glob(saving_at + "/*"):
    if file.endswith(".parquet") and "no_distance" in file:
        fingerprints += pd.read_parquet(file).sum(axis=1)
fingerprint = fingerprints.cumsum() / fingerprints.sum()
y = fingerprint.values
result = fit.ps_fingerprint_cdf_fit_1f(
    x, y, maxiter=2000, popsize=50, disp=False, rng=rng
)
np.save(saving_at + "\\" + r"fitting_parameters\\no_distance.npy", result.x)

: 